# SMB Ad Manager — full training pipeline (SFT + GRPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Falgunisharma72/smb-ad-manager/blob/main/notebooks/train_smb_ads.ipynb)

**Reproduces the trained 1.5B agent in ~60 minutes on a Colab Pro L4.**

This notebook runs the full pipeline:

1. **Setup** — clone the repo, install deps
2. **Login** — HuggingFace + Weights & Biases
3. **Stage 1: SFT warm-start** — 100 hand-crafted examples, 3 epochs, ~5 min
4. **Push SFT adapter to HF Hub**
5. **Stage 2: GRPO refinement** — 200 steps, group size 2, ~50 min, reward 0.41 → 0.71
6. **Push GRPO adapter to HF Hub**
7. **Pre-flight inference test** — verify the trained agent produces valid env actions

**Defaults are set for Qwen 2.5 1.5B Instruct** (the configuration that converged cleanly to +73% reward). To run the 3B scaling study instead, see the comments in Cell C and Cell E.

**Hardware**: Colab Pro L4 (24 GB) is sufficient. T4 will OOM on the 3B variant; A100 will be ~2× faster but is not required.

## Cell A — Setup (GPU check + clone + install) · ~3 min

In [ ]:
# GPU sanity check
!nvidia-smi | head -8

# Clone the repo
import os
if not os.path.exists("/content/smb-ad-manager"):
    !cd /content && git clone https://github.com/Falgunisharma72/smb-ad-manager.git
else:
    !cd /content/smb-ad-manager && git pull

%cd /content/smb-ad-manager

# Install training deps
!pip install -q "transformers>=4.46" "trl>=0.12" "peft>=0.13" "datasets>=3.0" \
    "bitsandbytes>=0.43" "accelerate>=1.0" "wandb" "huggingface_hub"

import torch, transformers, trl, peft
print(f"\ntorch: {torch.__version__}, cuda: {torch.cuda.is_available()}")
print(f"transformers: {transformers.__version__}, trl: {trl.__version__}, peft: {peft.__version__}")

## Cell B — Login (HuggingFace + W&B) · ~1 min

You need:
- A HuggingFace token with **write** access (for pushing trained adapters to your Hub)
- A W&B API key (for experiment tracking — optional but recommended)

Both prompts will appear inline.

In [ ]:
from huggingface_hub import login as hf_login
hf_login()

import wandb
wandb.login()

## Cell C — Stage 1: SFT warm-start · ~5 min on L4 (1.5B) · ~25 min (3B)

Trains a LoRA adapter on top of `Qwen/Qwen2.5-1.5B-Instruct` using 100 hand-crafted Meta Ads examples.

**To run the 3B scaling study instead**, run the patch cell below this one *before* running the SFT script. (It rewrites the model name + dtype settings in place.)

In [ ]:
# ─── Default: 1.5B SFT ───
# This trains the model that converged to +73% reward.
# Expected: loss 2.31 → ~0.17, token accuracy ~95%, ~5 min on L4.

!python -u training/sft_warm_start_nounsloth.py 2>&1 | tee /content/sft_run.log | tail -50

In [ ]:
# ─── OPTIONAL: switch to 3B before running SFT ───
# Run this cell INSTEAD OF the cell above if you want the 3B scaling study.
# Note: 3B SFT converges fine but GRPO collapses — see SUBMISSION.md scaling study.
#
# patches = [
#     ("Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"),
#     ("bnb_4bit_compute_dtype=torch.float16", "bnb_4bit_compute_dtype=torch.bfloat16"),
#     ("torch_dtype=torch.float16", "torch_dtype=torch.bfloat16"),
#     ("fp16=True", "bf16=True"),
#     ('OUTPUT_DIR = "./checkpoints/sft_adapter"', 'OUTPUT_DIR = "./checkpoints/sft_adapter_3b"'),
# ]
# path = "training/sft_warm_start_nounsloth.py"
# with open(path) as f:
#     src = f.read()
# for old, new in patches:
#     src = src.replace(old, new)
# with open(path, "w") as f:
#     f.write(src)
# print("✓ patched for 3B")

## Cell D — Push SFT adapter to HF Hub · ~30 sec

**Important:** push immediately after SFT completes. Colab runtimes can disconnect, and re-running SFT costs ~5-25 minutes you don't need to lose.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

# CHANGE this to your HF username
HF_USER = "Falgunisharma"
REPO = f"{HF_USER}/smb-ad-manager-sft"   # use "-sft-3b" if you ran the 3B variant
ADAPTER_DIR = "/content/smb-ad-manager/checkpoints/sft_adapter"  # "sft_adapter_3b" for 3B

api.create_repo(REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO,
    repo_type="model",
    commit_message="SFT warm-start adapter",
)
print(f"\n✓ Pushed to https://huggingface.co/{REPO}")

## Cell E — Stage 2: GRPO refinement · ~50 min on L4 (1.5B)

The GRPO reward function calls our deployed HF Space on every rollout. The agent is being scored against the same artifact judges run.

**What to watch in the W&B logs:**
- `reward` should climb past 0.50 by step 50, and toward 0.70 by step 200
- `reward_std` should stay above 0.05 (variance = learning signal)
- `frac_reward_zero_std` should stay below 0.5

**To run the 3B anti-collapse variant** (research artifact, did not converge), use `train_grpo_3b_v2.py` instead. The 3B story is in `SUBMISSION.md`.

In [ ]:
import os
os.environ["SPACE_URL"] = "https://falgunisharma-smb-ad-manager.hf.space"

# Default: 1.5B GRPO (the one that converged to +73%)
!python -u training/train_grpo_vanilla.py 2>&1 | tee /content/grpo_run.log

# To run 3B v2 (anti-collapse research run) instead, comment the line above and uncomment:
# !python -u training/train_grpo_3b_v2.py 2>&1 | tee /content/grpo_3b_v2_run.log

## Cell F — Push GRPO adapter to HF Hub · ~30 sec

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

HF_USER = "Falgunisharma"
REPO = f"{HF_USER}/smb-ad-manager-grpo"   # use "-grpo-3b-v2" if you ran the 3B variant
ADAPTER_DIR = "/content/smb-ad-manager/checkpoints/grpo_final"  # or "grpo_final_3b_v2"

api.create_repo(REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO,
    repo_type="model",
    commit_message="GRPO RL-refined adapter",
)
print(f"\n✓ Pushed to https://huggingface.co/{REPO}")

## Cell G — Pre-flight inference test · ~5 min

Generates 20 completions from the just-trained adapter, scores each against the live env, and prints a histogram.

**Why this matters:** before committing to another expensive training run, this test catches model-output bugs in 5 minutes that would otherwise cost 60+ minutes to discover. We used exactly this technique to diagnose the 3B over-generalization failure (see `SUBMISSION.md` scaling study).

**What success looks like for 1.5B GRPO:** at least 5 / 20 samples in the `>0.5` bucket, mean env score > 0.40, no hallucinated tools.

In [ ]:
import json, re, requests, urllib3
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
urllib3.disable_warnings()

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # use "Qwen/Qwen2.5-3B-Instruct" for 3B
ADAPTER_DIR = "./checkpoints/grpo_final"     # use "grpo_final_3b_v2" for 3B
SPACE_URL = "https://falgunisharma-smb-ad-manager.hf.space"

SESSION = requests.Session(); SESSION.verify = False
_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def call_space(path, payload):
    try:
        r = SESSION.post(f"{SPACE_URL}{path}", json=payload, timeout=20)
        return r.json() if r.ok else None
    except: return None

def extract(text):
    m = _JSON_RE.search(text.strip())
    if not m: return None
    try: return json.loads(m.group(0))
    except: return None

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

env_scores = []
for seed in range(20):
    obs = call_space("/reset", {"task_id": "easy", "seed": seed})
    if not obs:
        env_scores.append(("no_obs", 0.0)); continue
    smb = obs.get("smb_profile", {})
    user_msg = (
        f"Business: {smb.get('name')} ({smb.get('industry')})\n"
        f"Budget remaining: INR {obs.get('total_budget_remaining_inr', 0):.0f}\n"
        f"Active campaigns: {len(obs.get('active_campaigns', []))}\n"
        f"Latest metrics: {json.dumps(obs.get('latest_metrics', {}))}\n"
        "Return a JSON action like {\"tool\": \"...\", \"args\": {...}, \"reasoning\": \"...\"}"
    )
    msgs = [
        {"role": "system", "content": "You are an AI Ad Manager. Return a JSON action."},
        {"role": "user", "content": user_msg},
    ]
    prompt_text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256, temperature=1.0, do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    text = tok.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    action = extract(text)
    if not action or not isinstance(action, dict) or "tool" not in action:
        env_scores.append(("invalid", 0.0)); continue
    s = call_space("/step", {"action": action})
    score = float(s.get("reward", {}).get("score", 0.0)) if s else -1.0
    env_scores.append((action.get("tool", "?"), score))

print("\n=== Env score distribution (20 samples from trained agent) ===")
buckets = {">0.5": 0, "0.35-0.5": 0, "0.2-0.35": 0, "0.05-0.2": 0, "0-0.05": 0, "invalid/422": 0}
for tool, s in env_scores:
    if tool in ("invalid", "no_obs") or s < 0: buckets["invalid/422"] += 1
    elif s > 0.5: buckets[">0.5"] += 1
    elif s >= 0.35: buckets["0.35-0.5"] += 1
    elif s >= 0.2: buckets["0.2-0.35"] += 1
    elif s >= 0.05: buckets["0.05-0.2"] += 1
    else: buckets["0-0.05"] += 1
for k, v in buckets.items():
    print(f"  {k:14s} {'█' * v}{' ' * (20-v)} {v}/20")
valid = [s for _, s in env_scores if s >= 0]
print(f"\n  Mean env score: {sum(valid)/max(len(valid),1):.3f}")
print(f"  Tools used:     {set(t for t, _ in env_scores)}")

## Done — what now?

If everything ran cleanly:

- ✓ Your trained adapter is on HF Hub, ready to load with `PeftModel.from_pretrained`
- ✓ Training metrics are public on your W&B project
- ✓ The pre-flight test confirmed your agent produces valid env actions

**Next steps to play with:**

1. **Open the live demo** — <https://smb-ad-manager.vercel.app> · login `admin` / `hackathon2026`. The `/founder` page replays trained-model trajectories; the `/playground` page lets you act *as* the agent against the same env.
2. **Read the full writeup** — [SUBMISSION.md](https://github.com/Falgunisharma72/smb-ad-manager/blob/main/SUBMISSION.md) covers the reward design, the 3B scaling-study failure, and the proposed fixes.
3. **Try the 3B variant** — uncomment the patch cells above. The 3B scaling study is documented as a research finding in `SUBMISSION.md`; reproducing it takes ~90 minutes.

MIT licensed. Fork freely.